In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

# ===== RUTAS =====
base_dir = Path(r"C:\Users\juanb\Desktop\IronHack\Proyecto individual\proyecto-analysis-gaming\Limpieza de datos y scrapping")
output_dir = Path(r"C:\Users\juanb\Desktop\IronHack\Proyecto individual\proyecto-analysis-gaming\Tablas\H2")
output_dir.mkdir(parents=True, exist_ok=True)

# ===== CARGA =====
h1_file = Path(r"C:\Users\juanb\Desktop\IronHack\Proyecto individual\proyecto-analysis-gaming\Tablas\H1") / "h1_dataset_completo.csv"
df = pd.read_csv(h1_file)

print("="*60)
print("ANÁLISIS HIPÓTESIS 2: MARCA vs RESEÑAS")
print("="*60)

# ===== ANÁLISIS POR MARCA (GLOBAL) =====
print("\n📊 ANÁLISIS GLOBAL POR MARCA")
print("="*60)

marca_stats = df.groupby('brand').agg(
    n_productos=('product_name', 'count'),
    n_top10=('top_10_popularity', lambda x: x.sum()),
    pct_top10=('top_10_popularity', lambda x: (x.sum() / len(x) * 100)),
    rating_promedio=('rating', 'mean'),
    rating_mediana=('rating', 'median'),
    popularidad_promedio=('review_count', 'mean'),
    popularidad_mediana=('review_count', 'median'),
).reset_index()

# Calcular solo para marcas con al menos 3 productos
marca_stats = marca_stats[marca_stats['n_productos'] >= 3].copy()
marca_stats = marca_stats.sort_values('pct_top10', ascending=False)

print("\nTop 10 marcas por % de productos en Top 10% popularidad:")
print(marca_stats[['brand', 'n_productos', 'n_top10', 'pct_top10', 'rating_promedio']].head(10).to_string(index=False))

print("\n\nTop 10 marcas por rating promedio:")
marca_stats_rating = marca_stats.sort_values('rating_promedio', ascending=False)
print(marca_stats_rating[['brand', 'n_productos', 'pct_top10', 'rating_promedio']].head(10).to_string(index=False))

# ===== COMPARACIÓN: MARCAS POPULARES vs MEJOR VALORADAS =====
print("\n" + "="*60)
print("🔍 COMPARACIÓN: ¿COINCIDEN LAS MARCAS POPULARES CON LAS MEJOR VALORADAS?")
print("="*60)

# Top 5 marcas por popularidad (% en top 10)
top_populares = set(marca_stats.nlargest(5, 'pct_top10')['brand'])
# Top 5 marcas por rating
top_valoradas = set(marca_stats.nlargest(5, 'rating_promedio')['brand'])

print(f"\n✅ Top 5 marcas MÁS POPULARES (% en top 10):")
for marca in marca_stats.nlargest(5, 'pct_top10')['brand']:
    row = marca_stats[marca_stats['brand'] == marca].iloc[0]
    print(f"  • {marca}: {row['pct_top10']:.1f}% en top 10 | Rating: {row['rating_promedio']:.2f}")

print(f"\n⭐ Top 5 marcas MEJOR VALORADAS:")
for marca in marca_stats.nlargest(5, 'rating_promedio')['brand']:
    row = marca_stats[marca_stats['brand'] == marca].iloc[0]
    print(f"  • {marca}: Rating {row['rating_promedio']:.2f} | {row['pct_top10']:.1f}% en top 10")

coincidencias = top_populares.intersection(top_valoradas)
print(f"\n📌 Marcas que aparecen en AMBOS tops: {coincidencias if coincidencias else 'NINGUNA'}")
print(f"📌 Coincidencia: {len(coincidencias)}/5 marcas ({len(coincidencias)/5*100:.0f}%)")

# ===== ANÁLISIS POR SUBCATEGORÍA =====
print("\n" + "="*60)
print("📦 ANÁLISIS POR SUBCATEGORÍA")
print("="*60)

for subcat in sorted(df['subcategory'].unique()):
    print(f"\n{'='*60}")
    print(f"📦 {subcat.upper()}")
    print(f"{'='*60}")
    
    subcat_data = df[df['subcategory'] == subcat]
    
    marca_subcat = subcat_data.groupby('brand').agg(
        n_productos=('product_name', 'count'),
        n_top10=('top_10_popularity', lambda x: x.sum()),
        pct_top10=('top_10_popularity', lambda x: (x.sum() / len(x) * 100) if len(x) > 0 else 0),
        rating_promedio=('rating', 'mean'),
        popularidad_promedio=('review_count', 'mean'),
    ).reset_index()
    
    # Filtrar marcas con al menos 2 productos en esta subcategoría
    marca_subcat = marca_subcat[marca_subcat['n_productos'] >= 2].copy()
    
    if len(marca_subcat) == 0:
        print("  (No hay marcas con 2+ productos)")
        continue
    
    # Top 3 por popularidad
    print("\n  Top 3 marcas MÁS POPULARES:")
    for _, row in marca_subcat.nlargest(3, 'pct_top10').iterrows():
        print(f"    • {row['brand']}: {row['pct_top10']:.0f}% en top 10 | Rating: {row['rating_promedio']:.2f}")
    
    # Top 3 por rating
    print("\n  Top 3 marcas MEJOR VALORADAS:")
    for _, row in marca_subcat.nlargest(3, 'rating_promedio').iterrows():
        print(f"    • {row['brand']}: Rating {row['rating_promedio']:.2f} | {row['pct_top10']:.0f}% en top 10")

# ===== ANÁLISIS POR GAMA DE PRECIO =====
print("\n" + "="*60)
print("💰 ANÁLISIS POR GAMA DE PRECIO")
print("="*60)

for tier in ['baja', 'media', 'alta']:
    print(f"\n{'='*60}")
    print(f"💰 GAMA {tier.upper()}")
    print(f"{'='*60}")
    
    tier_data = df[df['price_tier'] == tier]
    
    marca_tier = tier_data.groupby('brand').agg(
        n_productos=('product_name', 'count'),
        n_top10=('top_10_popularity', lambda x: x.sum()),
        pct_top10=('top_10_popularity', lambda x: (x.sum() / len(x) * 100) if len(x) > 0 else 0),
        rating_promedio=('rating', 'mean'),
        popularidad_promedio=('review_count', 'mean'),
    ).reset_index()
    
    marca_tier = marca_tier[marca_tier['n_productos'] >= 2].copy()
    
    if len(marca_tier) == 0:
        print("  (No hay marcas con 2+ productos en esta gama)")
        continue
    
    print("\n  Top 5 marcas MÁS POPULARES:")
    for _, row in marca_tier.nlargest(5, 'pct_top10').iterrows():
        print(f"    • {row['brand']}: {row['pct_top10']:.0f}% en top 10 | Rating: {row['rating_promedio']:.2f} | {row['n_productos']} productos")
    
    print("\n  Top 5 marcas MEJOR VALORADAS:")
    for _, row in marca_tier.nlargest(5, 'rating_promedio').iterrows():
        print(f"    • {row['brand']}: Rating {row['rating_promedio']:.2f} | {row['pct_top10']:.0f}% en top 10 | {row['n_productos']} productos")

# ===== CORRELACIÓN: RATING vs POPULARIDAD =====
print("\n" + "="*60)
print("📈 CORRELACIÓN: RATING vs POPULARIDAD (por marca)")
print("="*60)

# Solo marcas con 3+ productos
marca_corr = df.groupby('brand').agg(
    n_productos=('product_name', 'count'),
    rating_promedio=('rating', 'mean'),
    popularidad_promedio=('review_count', 'mean'),
).reset_index()
marca_corr = marca_corr[marca_corr['n_productos'] >= 3]

correlacion = marca_corr['rating_promedio'].corr(marca_corr['popularidad_promedio'])
print(f"\nCorrelación entre Rating Promedio y Popularidad Promedio (por marca):")
print(f"  Coeficiente de correlación: {correlacion:.3f}")

if abs(correlacion) < 0.3:
    print(f"  Interpretación: CORRELACIÓN DÉBIL (la marca importa más que el rating)")
elif abs(correlacion) < 0.7:
    print(f"  Interpretación: CORRELACIÓN MODERADA")
else:
    print(f"  Interpretación: CORRELACIÓN FUERTE")

# ===== GUARDAR RESULTADOS =====
marca_stats.to_csv(output_dir / "h2_analisis_marcas_global.csv", index=False, encoding="utf-8-sig")
df.to_csv(output_dir / "h2_dataset_completo.csv", index=False, encoding="utf-8-sig")

print(f"\n{'='*60}")
print(f"✅ Archivos guardados en: {output_dir}")
print(f"{'='*60}")

ANÁLISIS HIPÓTESIS 2: MARCA vs RESEÑAS

📊 ANÁLISIS GLOBAL POR MARCA

Top 10 marcas por % de productos en Top 10% popularidad:
    brand  n_productos  n_top10  pct_top10  rating_promedio
      MSI           16        5  31.250000         4.650000
      AOC           11        3  27.272727         4.590909
 Logitech           22        6  27.272727         4.645455
     Sony           11        3  27.272727         4.681818
  Tempest           20        5  25.000000         4.410000
Microsoft            9        1  11.111111         4.666667
    Razer           18        1   5.555556         4.605556
     Acer            6        0   0.000000         4.683333
       HP            4        0   0.000000         4.575000
  Gamesir            3        0   0.000000         4.533333


Top 10 marcas por rating promedio:
    brand  n_productos  pct_top10  rating_promedio
   Indeca            3   0.000000         4.900000
     Asus            3   0.000000         4.733333
   HyperX            4  